In [2]:
# inventory_analysis_end_to_end.py
# End-to-end Inventory Data Analysis (demo-ready if files missing)
# version 1
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy.stats import norm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor

# Optional interactive viz
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

sns.set(style='whitegrid')



# 1) Settings & filenames

DATA_DIR = "data"
# common filenames used in the project pdf examples
FILES = {
    "purchase_price": "2017PurchasePricesDec.csv",
    "beg_inv": "BegInvFINAL12312016.csv",
    "end_inv": "EndInvFINAL12312016.csv",
    "invoice": "InvoicePurchases12312016.csv",
    "purchases": "PurchasesFINAL12312016.csv",
    "sales": "SalesFINAL12312016.csv"
}

# Parameters (tune for your company)
ANNUAL_CARRYING_RATE = 0.25   # 25% carrying cost default (annual)
DEFAULT_ORDER_COST = 100.0    # typical fixed ordering cost per order
DEFAULT_SERVICE_LEVEL = 0.95  # 95% service level
FORECAST_HORIZON_DAYS = 90    # horizon to forecast (days)
TOP_SKU_FORECAST = 50         # only forecast top N SKUs (speeds up demo)



# 2) Data ingestion with fallback synthetic data

def load_csv_if_exists(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return None

def generate_demo_data():
    """Generates small synthetic inventory/purchase/sales datasets so pipeline runs."""
    rng = np.random.default_rng(42)
    inventory_ids = [f"SKU{str(i).zfill(4)}" for i in range(1, 101)]
    stores = [f"W{i}" for i in range(1,6)]

    # Beginning inventory snapshot
    beg = []
    for sku in inventory_ids:
        for store in stores:
            beg.append({
                "InventoryId": sku,
                "Store": store,
                "City": "DemoCity",
                "Brand": rng.integers(1,100),
                "Description": f"Product {sku}",
                "Size": "unit",
                "onHand": int(rng.integers(0, 200)),
                "Price": float(round(rng.uniform(5, 200),2)),
                "startDate": "2016-01-01"
            })
    beg = pd.DataFrame(beg)

    # Sales: random daily sales for 365 days for selected SKUs
    dates = pd.date_range("2016-01-01", periods=365)
    sales = []
    for sku in inventory_ids:
        base = rng.integers(0, 10)
        season = (np.sin(np.linspace(0, 6.28, len(dates))) + 1) * rng.uniform(0.2, 2)
        noise = rng.normal(0, 1, len(dates))
        qty = np.maximum(0, (base * season + noise).round().astype(int))
        price = float(round(rng.uniform(10, 150),2))
        for dt, q in zip(dates, qty):
            if q > 0 and rng.random() < 0.2:  # only some days have orders
                sales.append({
                    "InventoryId": sku,
                    "Store": stores[rng.integers(0, len(stores))],
                    "Brand": rng.integers(1,100),
                    "Description": f"Product {sku}",
                    "Size": "unit",
                    "SalesQuantity": int(q),
                    "SalesDollars": float(round(q * price,2)),
                    "SalesPrice": price,
                    "SalesDate": dt.strftime("%Y-%m-%d"),
                    "Volume": 1,
                    "Classification": 1,
                    "ExciseTax": 0.0,
                    "VendorNo": rng.integers(1000,2000),
                    "VendorName": f"Vendor_{rng.integers(1,20)}"
                })
    sales = pd.DataFrame(sales)

    # Purchases (order arrivals)
    purchases = []
    for sku in inventory_ids:
        orders = rng.integers(1, 8)
        last_date = pd.Timestamp("2016-01-01")
        for _ in range(orders):
            lead = int(rng.integers(3, 21))
            qty = int(rng.integers(10, 500))
            order_date = last_date + pd.Timedelta(days=int(rng.integers(7, 30)))
            receive_date = order_date + pd.Timedelta(days=lead)
            purchases.append({
                "InventoryId": sku,
                "OrderDate": order_date.strftime("%Y-%m-%d"),
                "ReceiptDate": receive_date.strftime("%Y-%m-%d"),
                "Quantity": qty,
                "UnitCost": float(round(rng.uniform(5, 120),2)),
                "VendorName": f"Vendor_{rng.integers(1,20)}"
            })
            last_date = order_date
    purchases = pd.DataFrame(purchases)

    # Invoice / invoice-like info (ordering cost proxy)
    invoice = pd.DataFrame([{"VendorNumber": 100, "VendorName": "Vendor_1", "InvoiceDate": "2016-01-04", "PONumber": "PO1", "PODate": "2015-12-21", "PayDate": "2016-02-16", "Quantity": 6, "Dollars": 214.26, "Freight": 3.47}])


    return {
        "BegInv": beg,
        "Sales": sales,
        "Purchases": purchases,
        "Invoice": invoice,
        "EndInv": beg.copy(),  # simple
        "PurchasePrice": None
    }

def load_data(data_dir=DATA_DIR):
    data = {}
    # try loading each file
    for key, fn in FILES.items():
        full = os.path.join(data_dir, fn)
        df = load_csv_if_exists(full)
        if df is None:
            data = None
            break
        data[key] = df
    if data is None:
        print("One or more dataset files not found in 'data/'. Running with generated demo data.")
        return generate_demo_data()
    # rename keys similar to demo return for consistency
    return {
        "BegInv": data["beg_inv"],
        "EndInv": data["end_inv"],
        "Sales": data["sales"],
        "Purchases": data["purchases"],
        "Invoice": data["invoice"],
        "PurchasePrice": data.get("purchase_price")
    }

# Load
datasets = load_data()



# 3) Quick sanity checks & cleaning

def sanity_checks_and_clean(datasets):
    # Basic presence checks
    for k, df in datasets.items():
        if df is None:
            continue
        print(f"{k}: shape {df.shape}")
        print(df.dtypes.value_counts().to_dict())
        print(df.head(1))
    # Convert dates
    if "Sales" in datasets and datasets["Sales"] is not None:
        s = datasets["Sales"]
        for col in ["SalesDate", "OrderDate", "ReceiptDate"]:
            if col in s.columns:
                s[col] = pd.to_datetime(s[col], errors='coerce')
        # if SalesDate is string in demo data, convert:
        if "SalesDate" in s.columns and s["SalesDate"].dtype == object:
            s["SalesDate"] = pd.to_datetime(s["SalesDate"], errors='coerce')
        datasets["Sales"] = s

    if "Purchases" in datasets and datasets["Purchases"] is not None:
        p = datasets["Purchases"]
        for col in ["OrderDate", "ReceiptDate", "PODate"]:
            if col in p.columns:
                p[col] = pd.to_datetime(p[col], errors='coerce')
        datasets["Purchases"] = p

    return datasets

datasets = sanity_checks_and_clean(datasets)



# 4) Aggregate sales and compute basic KPIs

def daily_sales_aggregate(sales_df):
    # expects SalesDate column
    sales_df = sales_df.copy()
    sales_df['SalesDate'] = pd.to_datetime(sales_df['SalesDate'])
    daily = sales_df.groupby(['InventoryId', 'SalesDate']).agg({
        'SalesQuantity':'sum',
        'SalesDollars':'sum',
        'SalesPrice':'mean'
    }).reset_index()
    # fill dates with zeros per SKU for continuous time series if needed later
    return daily

daily_sales = daily_sales_aggregate(datasets['Sales'])
print("Aggregated daily sales sample:")
print(daily_sales.head())



# 5) Compute lead times from purchases (if available), else default

def compute_lead_times(purchases_df):
    if purchases_df is None or purchases_df.empty:
        return None
    p = purchases_df.copy()
    # expected columns: OrderDate, ReceiptDate
    if 'OrderDate' in p.columns and 'ReceiptDate' in p.columns:
        p = p.dropna(subset=['OrderDate', 'ReceiptDate'])
        p['lead_days'] = (pd.to_datetime(p['ReceiptDate']) - pd.to_datetime(p['OrderDate'])).dt.days
        lt = p.groupby('InventoryId')['lead_days'].agg(['mean', 'median', 'std']).reset_index()
        lt.rename(columns={'mean':'lead_mean','median':'lead_median','std':'lead_std'}, inplace=True)
        return lt
    return None

lead_times = compute_lead_times(datasets.get('Purchases'))
if lead_times is None:
    print("Lead time data not available; will use defaults.")
else:
    print("Lead times sample:")
    print(lead_times.head())



# 6) Demand forecasting (per SKU) using Holt-Winters (ETS)

def forecast_sku_demand(daily_sales, sku, forecast_days=FORECAST_HORIZON_DAYS):
    """Return forecast series for sku: mean forecast and sample variance estimate."""
    df = daily_sales[daily_sales['InventoryId']==sku].set_index('SalesDate').sort_index()
    # reindex to continuous daily dates
    idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    df = df.reindex(idx, fill_value=0)
    series = df['SalesQuantity'].astype(float)
    # If series mostly zeros or short, return naive average forecast
    if series.sum() < 5 or len(series.dropna()) < 30:
        mu = series.mean()
        forecast = pd.Series([mu]*forecast_days, index=pd.date_range(series.index.max()+pd.Timedelta(days=1), periods=forecast_days))
        var = max(1.0, series.var())
        return forecast, var
    try:
        model = ExponentialSmoothing(series, trend='add', seasonal=None, initialization_method='estimated')
        fit = model.fit(optimized=True)
        fc = fit.forecast(forecast_days)
        # estimate variance using residuals
        resid_var = np.var(fit.resid.dropna()) if len(fit.resid.dropna())>1 else max(1.0, series.var())
        return fc, resid_var
    except Exception as e:
        # fallback to rolling mean
        print(f"Holt-Winters failed for {sku}: {e}. Using mean fallback.")
        mu = series.mean()
        fc = pd.Series([mu]*forecast_days, index=pd.date_range(series.index.max()+pd.Timedelta(days=1), periods=forecast_days))
        return fc, max(1.0, series.var())

# Forecast top SKUs by recent sales volume
sku_total = daily_sales.groupby('InventoryId')['SalesQuantity'].sum().sort_values(ascending=False)
top_skus = sku_total.head(TOP_SKU_FORECAST).index.tolist()

forecasts = {}
for sku in top_skus:
    fc, var = forecast_sku_demand(daily_sales, sku, forecast_days=FORECAST_HORIZON_DAYS)
    forecasts[sku] = {"forecast":fc, "variance": var, "mean_daily": float(fc.mean())}

print(f"Forecasted {len(forecasts)} SKUs (top {TOP_SKU_FORECAST}). Example SKU: {list(forecasts.keys())[0]} -> mean daily {forecasts[list(forecasts.keys())[0]]['mean_daily']:.2f}")



# 7) EOQ & Reorder Point calculations

def compute_eoq(D_annual, Co, Ch_per_unit_per_year):
    # D_annual: annual demand (units)
    # Co: ordering cost per order
    # Ch: holding cost per unit per year
    if D_annual <= 0 or Ch_per_unit_per_year <= 0:
        return np.nan
    return np.sqrt((2 * D_annual * Co) / Ch_per_unit_per_year)

def compute_reorder_point(avg_daily_demand, lead_time_days, demand_variance_daily, service_level=DEFAULT_SERVICE_LEVEL):
    z = norm.ppf(service_level)
    safety_stock = z * np.sqrt(lead_time_days * demand_variance_daily)
    rop = lead_time_days * avg_daily_demand + safety_stock
    return max(0.0, rop), safety_stock

# Build EOQ / ROP per SKU table (for SKUs we forecasted)
eoq_rows = []
for sku in top_skus:
    mean_daily = forecasts[sku]['mean_daily']          # units/day
    var_daily = forecasts[sku]['variance']             # (units^2)
    D_annual = mean_daily * 365.0
    # unit cost from PurchasePrice or SalesPrice fallback
    unit_cost = None
    # try to fetch unit cost from purchases or sales
    if 'Purchases' in datasets and datasets['Purchases'] is not None and 'UnitCost' in datasets['Purchases'].columns:
        tmp = datasets['Purchases'][datasets['Purchases']['InventoryId']==sku]
        if not tmp.empty:
            unit_cost = tmp['UnitCost'].mean()
    if unit_cost is None:
        tmp = datasets['Sales'][datasets['Sales']['InventoryId']==sku]
        if not tmp.empty and 'SalesPrice' in tmp.columns:
            unit_cost = tmp['SalesPrice'].mean()
    if unit_cost is None:
        unit_cost = 10.0  # fallback

    Ch = ANNUAL_CARRYING_RATE * float(unit_cost)   # holding cost per unit per year
    Co = DEFAULT_ORDER_COST

    eoq = compute_eoq(D_annual, Co, Ch)
    # lead time: if lead_times table exists, use sku mean else use default 14 days
    lead_time_days = 14
    if lead_times is not None and sku in list(lead_times['InventoryId'].astype(str)):
        lt_row = lead_times[lead_times['InventoryId']==sku]
        if not lt_row.empty and not np.isnan(lt_row['lead_mean'].values[0]):
            lead_time_days = max(1, int(round(float(lt_row['lead_mean'].values[0]))))

    rop, safety_stock = compute_reorder_point(mean_daily, lead_time_days, var_daily, service_level=DEFAULT_SERVICE_LEVEL)

    eoq_rows.append({
        "InventoryId": sku,
        "mean_daily_demand": mean_daily,
        "annual_demand_est": D_annual,
        "unit_cost_est": float(unit_cost),
        "holding_cost_per_unit_year": Ch,
        "order_cost": Co,
        "EOQ_units": float(np.round(eoq, 2)) if not np.isnan(eoq) else np.nan,
        "lead_time_days": lead_time_days,
        "ROP_units": float(np.round(rop,2)),
        "safety_stock": float(np.round(safety_stock,2))
    })

eoq_df = pd.DataFrame(eoq_rows).sort_values("annual_demand_est", ascending=False)
eoq_df.to_csv("eoq_reorder_point_per_sku.csv", index=False)
print("Saved EOQ / ROP table to eoq_reorder_point_per_sku.csv")
print(eoq_df.head())



# 8) ABC analysis (by annual consumption value)

def abc_classification(sku_annual_df):
    sku_annual_df = sku_annual_df.copy()
    sku_annual_df['annual_value'] = sku_annual_df['annual_demand_est'] * sku_annual_df['unit_cost_est']
    sku_annual_df = sku_annual_df.sort_values('annual_value', ascending=False)
    sku_annual_df['cum_value'] = sku_annual_df['annual_value'].cumsum()
    total = sku_annual_df['annual_value'].sum()
    sku_annual_df['cum_pct'] = sku_annual_df['cum_value'] / total
    # classify: A: top ~70% value, B: next ~20%, C: last ~10% (adjust if you want 80/15/5)
    def cls(pct):
        if pct <= 0.7:
            return 'A'
        elif pct <= 0.9:
            return 'B'
        else:
            return 'C'
    sku_annual_df['ABC'] = sku_annual_df['cum_pct'].apply(cls)
    return sku_annual_df

abc_df = abc_classification(eoq_df)
abc_df.to_csv("abc_eoq.csv", index=False)
print("ABC classification sample:")
print(abc_df[['InventoryId','annual_value','cum_pct','ABC']].head())



# 9) Inventory cost & turnover metrics

def compute_inventory_metrics(beg_inv, end_inv, sales_df):
    # rough metrics: average inventory per SKU, turnover = cost_of_goods_sold / avg_inventory
    # For demonstration we assume unit_cost = SalesPrice average
    avg_inv = None
    if beg_inv is not None and end_inv is not None:
        # compute avg onHand per sku across all warehouses
        beg = beg_inv.groupby('InventoryId')['onHand'].sum().rename('beg_onhand')
        end = end_inv.groupby('InventoryId')['onHand'].sum().rename('end_onhand')
        avg = pd.concat([beg, end], axis=1).fillna(0)
        avg['avg_onhand'] = (avg['beg_onhand'] + avg['end_onhand']) / 2.0
        avg_inv = avg.reset_index()
    # COGS-ish: sum of sales quantity * avg price (approx)
    sales_agg = sales_df.groupby('InventoryId').agg({'SalesQuantity':'sum','SalesDollars':'sum','SalesPrice':'mean'}).reset_index()
    merged = sales_agg.merge(avg_inv, how='left', on='InventoryId') if avg_inv is not None else sales_agg.copy()
    # estimate avg inventory value
    merged['avg_inventory_value'] = merged['avg_onhand'] * merged['SalesPrice']
    merged['turnover'] = merged.apply(lambda r: (r['SalesDollars'] / r['avg_inventory_value']) if r['avg_inventory_value']>0 else np.nan, axis=1)
    merged['days_of_inventory'] = merged.apply(lambda r: (365.0 / r['turnover']) if (r['turnover']>0 and not np.isnan(r['turnover'])) else np.nan, axis=1)
    return merged

metrics_df = compute_inventory_metrics(datasets.get('BegInv'), datasets.get('EndInv'), datasets.get('Sales'))
metrics_df.to_csv("inventory_metrics.csv", index=False)
print("Inventory metrics sample:")
print(metrics_df.head())


# 10) Stockout / Excess identification (simple)
# Identify SKUs with high forecast but low on-hand

current_onhand = None
if datasets.get('EndInv') is not None:
    current_onhand = datasets['EndInv'].groupby('InventoryId')['onHand'].sum().rename('onHand').reset_index()
else:
    current_onhand = pd.DataFrame({"InventoryId": top_skus, "onHand": 0})

risk_rows = []
for sku in top_skus:
    mean_daily = forecasts[sku]['mean_daily']
    projected_30d = mean_daily * 30
    onhand = float(current_onhand[current_onhand['InventoryId']==sku]['onHand'].sum()) if not current_onhand[current_onhand['InventoryId']==sku].empty else 0.0
    gap = onhand - projected_30d
    risk_rows.append({"InventoryId": sku, "onHand": onhand, "proj_30d_demand": projected_30d, "onhand_minus_30d": gap, "ABC": abc_df.set_index('InventoryId').loc[sku]['ABC'] if sku in list(abc_df['InventoryId']) else 'C'})

risk_df = pd.DataFrame(risk_rows).sort_values("onhand_minus_30d")
risk_df.to_csv("stockout_risk.csv", index=False)
print("High risk SKUs (lowest onhand_minus_30d):")
print(risk_df.head(10))



# 11) Simple recommendations & prioritized actions

def recommend_actions(eoq_df, risk_df, abc_df, metrics_df):
    recs = []
    # For A items — tighter controls
    for _,row in abc_df.iterrows():
        sku = row['InventoryId']
        abc = row['ABC']
        eoq_row = eoq_df[eoq_df['InventoryId']==sku]
        risk_row = risk_df[risk_df['InventoryId']==sku]
        avg_daily = eoq_row['mean_daily_demand'].values[0] if not eoq_row.empty else 0
        rop = eoq_row['ROP_units'].values[0] if not eoq_row.empty else 0
        onhand = float(risk_row['onHand'].values[0]) if not risk_row.empty else 0
        if abc == 'A':
            policy = "Daily/continuous review. Consider safety stock increase and short lead-time suppliers."
        elif abc == 'B':
            policy = "Weekly review. Use EOQ orders; moderate safety stock."
        else:
            policy = "Periodic review monthly. Larger batch orders to reduce ordering cost."
        recs.append({"InventoryId": sku, "ABC": abc, "avg_daily": avg_daily, "ROP": rop, "onHand": onhand, "policy": policy})
    return pd.DataFrame(recs)

recommendations_df = recommend_actions(eoq_df, risk_df, abc_df, metrics_df)
recommendations_df.head(10).to_csv("recommendations.csv", index=False)
print("Saved sample recommendations to recommendations.csv")



# 12) Simple Plotly dashboard (if available)

if PLOTLY_AVAILABLE:
    # Top 20 risky SKUs
    top_risky = risk_df.sort_values("onhand_minus_30d").head(20)
    fig = px.bar(top_risky, x='InventoryId', y='onhand_minus_30d', color='ABC', title='Top 20 stockout risk (onHand - 30d forecast)')
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_html("dashboard_stockout_risk.html")
    print("Saved interactive dashboard to dashboard_stockout_risk.html (open in browser).")
else:
    # fallback matplotlib
    plt.figure(figsize=(12,5))
    sample = risk_df.sort_values("onhand_minus_30d").head(20)
    sns.barplot(x='InventoryId', y='onhand_minus_30d', hue='ABC', data=sample, dodge=False)
    plt.xticks(rotation=60)
    plt.title("Top 20 stockout risk (onHand - 30d forecast)")
    plt.tight_layout()
    plt.savefig("dashboard_stockout_risk.png", dpi=150)
    print("Plot saved as dashboard_stockout_risk.png (plotly not available).")


# 13) Save cleaned datasets (for deliverables)

# Save cleaned/aggregated artifacts

daily_sales.to_csv("cleaned_daily_sales.csv", index=False)
print("Saved cleaned_daily_sales.csv")

# Final prints
print("\n--- Pipeline completed ---")
print("Generated files:")
for f in ["eoq_reorder_point_per_sku.csv","abc_eoq.csv","inventory_metrics.csv","stockout_risk.csv","recommendations.csv","cleaned_daily_sales.csv"]:
    if os.path.exists(f):
        print("  -", f)
if PLOTLY_AVAILABLE and os.path.exists("dashboard_stockout_risk.html"):
    print("  - dashboard_stockout_risk.html")
elif os.path.exists("dashboard_stockout_risk.png"):
    print("  - dashboard_stockout_risk.png")

# End of script


One or more dataset files not found in 'data/'. Running with generated demo data.
BegInv: shape (500, 9)
{dtype('O'): 6, dtype('int64'): 2, dtype('float64'): 1}
  InventoryId Store      City  Brand      Description  Size  onHand  Price  \
0     SKU0001    W1  DemoCity      9  Product SKU0001  unit     154  90.58   

    startDate  
0  2016-01-01  
Sales: shape (5474, 14)
{dtype('O'): 6, dtype('int64'): 5, dtype('float64'): 3}
  InventoryId Store  Brand      Description  Size  SalesQuantity  \
0     SKU0001    W1     58  Product SKU0001  unit              8   

   SalesDollars  SalesPrice   SalesDate  Volume  Classification  ExciseTax  \
0        986.08      123.26  2016-01-02       1               1        0.0   

   VendorNo VendorName  
0      1651  Vendor_19  
Purchases: shape (372, 6)
{dtype('O'): 4, dtype('int64'): 1, dtype('float64'): 1}
  InventoryId   OrderDate ReceiptDate  Quantity  UnitCost VendorName
0     SKU0001  2016-01-14  2016-01-21       447     18.22  Vendor_15
Invoic

In [4]:

# ADVANCED INVENTORY OPTIMIZATION SYSTEM
#version 2
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
from scipy.optimize import minimize

import sqlite3

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except:
    PLOTLY_AVAILABLE = False


#  LOAD DATA


DATA_PATH = "amazon.csv"

if not os.path.exists(DATA_PATH):
    print("Dataset not found. Generating demo dataset...")
    np.random.seed(42)
    n = 300
    
    df = pd.DataFrame({
        "product_id": [f"P{i}" for i in range(n)],
        "product_name": [f"Product_{i}" for i in range(n)],
        "category": np.random.choice(["Electronics","Appliances","Accessories"], n),
        "discounted_price": np.random.uniform(100,1000,n),
        "actual_price": np.random.uniform(150,1200,n),
        "discount_percentage": np.random.uniform(5,40,n),
        "rating": np.random.uniform(2.5,5,n),
        "rating_count": np.random.randint(10,2000,n),
        "about_product": ["Demo"]*n,
        "user_id": np.random.randint(1000,5000,n),
        "user_name": ["User"]*n,
        "review_id": np.random.randint(10000,20000,n),
        "review_title": ["Good"]*n,
        "review_content": ["Nice product"]*n,
        "img_link": [""]*n,
        "product_link": [""]*n
    })
else:
    df = pd.read_csv(DATA_PATH)

print("Data Loaded:", df.shape)



# FEATURE ENGINEERING (FIXED VERSION)


# Clean numeric columns safely
numeric_cols = [
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating",
    "rating_count"
]

for col in numeric_cols:
    # Convert to string first
    df[col] = df[col].astype(str)
    
    # Remove currency symbols, commas, spaces
    df[col] = df[col].str.replace(r"[^\d.]", "", regex=True)
    
    # Convert to numeric
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing numeric values
df[numeric_cols] = df[numeric_cols].fillna(0)

# Now calculations will NOT fail
df["unit_cost"] = df["actual_price"]
df["margin"] = df["actual_price"] - df["discounted_price"]
df["price_diff"] = df["discount_percentage"]
df["demand_proxy"] = df["rating_count"]

# Simulate demand & variance
df["avg_daily_demand"] = df["demand_proxy"] / 365
df["demand_variance"] = df["avg_daily_demand"] * 0.5

# Simulate lead time
np.random.seed(42)
df["lead_time"] = np.random.randint(3, 20, len(df))



# MODEL COMPARISON (RMSE, MAE, MAPE)


def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred)/y_true))*100

features = ["discounted_price","actual_price","discount_percentage","rating","margin"]

X = df[features]
y = df["avg_daily_demand"]

tscv = TimeSeriesSplit(n_splits=5)

models = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    rmse_list, mae_list, mape_list = [], [], []
    
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        rmse_list.append(np.sqrt(mean_squared_error(y_test, preds)))
        mae_list.append(mean_absolute_error(y_test, preds))
        mape_list.append(mape(y_test, preds))
    
    results.append({
        "Model": name,
        "RMSE": np.mean(rmse_list),
        "MAE": np.mean(mae_list),
        "MAPE": np.mean(mape_list)
    })

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df)

# Train final model
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X, y)
df["forecast_demand"] = final_model.predict(X)


# EOQ + SAFETY STOCK (Monte Carlo)


CARRYING_RATE = 0.25
ORDER_COST = 50
SERVICE_LEVEL = 0.95
z = norm.ppf(SERVICE_LEVEL)

df["annual_demand"] = df["forecast_demand"] * 365
df["holding_cost"] = df["unit_cost"] * CARRYING_RATE

df["EOQ"] = np.sqrt((2 * df["annual_demand"] * ORDER_COST) / df["holding_cost"])

df["safety_stock"] = z * np.sqrt(df["lead_time"] * df["demand_variance"])
df["ROP"] = df["lead_time"] * df["forecast_demand"] + df["safety_stock"]


# OPTIMIZATION (Cost Minimization)


def total_cost(q, D, Co, Ch):
    return (D/q)*Co + (q/2)*Ch

df["optimized_order_qty"] = df.apply(
    lambda row: minimize(
        total_cost,
        x0=row["EOQ"],
        args=(row["annual_demand"], ORDER_COST, row["holding_cost"])
    ).x[0],
    axis=1
)


# ABC ANALYSIS


df["annual_value"] = df["annual_demand"] * df["unit_cost"]
df = df.sort_values("annual_value", ascending=False)
df["cum_pct"] = df["annual_value"].cumsum()/df["annual_value"].sum()

def abc(p):
    if p<=0.7: return "A"
    elif p<=0.9: return "B"
    else: return "C"

df["ABC"] = df["cum_pct"].apply(abc)


#  CLUSTERING


cluster_features = df[["annual_demand","unit_cost","lead_time"]]
scaler = StandardScaler()
scaled = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(scaled)


# RISK SCORING


df["risk_score"] = (
    df["demand_variance"]*0.4 +
    df["lead_time"]*0.3 +
    df["annual_value"]/df["annual_value"].max()*0.3
)

df = df.sort_values("risk_score", ascending=False)


# FINANCIAL IMPACT


df["current_cost"] = df["annual_demand"]/df["EOQ"]*ORDER_COST + df["EOQ"]/2*df["holding_cost"]
df["optimized_cost"] = df["annual_demand"]/df["optimized_order_qty"]*ORDER_COST + df["optimized_order_qty"]/2*df["holding_cost"]

df["savings"] = df["current_cost"] - df["optimized_cost"]

print("\nEstimated Total Savings:", df["savings"].sum())


#  REPLENISHMENT ENGINE


df["current_inventory"] = np.random.randint(0,200,len(df))

df["reorder_flag"] = df["current_inventory"] <= df["ROP"]

purchase_orders = df[df["reorder_flag"]==True][["product_id","optimized_order_qty"]]
purchase_orders.to_csv("purchase_orders.csv", index=False)


# SQL STORAGE


conn = sqlite3.connect("inventory_system.db")
df.to_sql("inventory_data", conn, if_exists="replace", index=False)
conn.close()


 # DASHBOARD EXPORT


if PLOTLY_AVAILABLE:
    fig = px.bar(df.head(20), x="product_id", y="risk_score",
                 title="Top 20 Risk Products")
    fig.write_html("dashboard.html")
    print("Dashboard saved as dashboard.html")


# SAVE FINAL OUTPUT


df.to_csv("final_inventory_analysis.csv", index=False)

print("\nPipeline Completed Successfully of version 2.")


Data Loaded: (1465, 16)

Model Comparison:
          Model        RMSE        MAE  MAPE
0  RandomForest  110.760742  65.791245   inf

Estimated Total Savings: 1.5234036254696548e-10
Dashboard saved as dashboard.html

Pipeline Completed Successfully of version 2.
